# Data Preprocessing and Feature Engineering Pipeline

In [1]:
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import VarianceThreshold
import lightgbm as lgb
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
folder_name = 'Data'

## Loading the dataset

In [3]:
df = pd.read_csv('./Data/fraud_oracle.csv')
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,21,Policy Holder,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,34,Policy Holder,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,47,Policy Holder,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,65,Third Party,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,27,Third Party,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


In [4]:
df.shape

(15420, 33)

In [5]:
df_copy = df.copy()

## Label Encoding for Categorical Columns

In [6]:
encoder = LabelEncoder()
columns_to_encode = ['AccidentArea', 'Sex', 'Fault', 'PoliceReportFiled', 'WitnessPresent', 'AgentType']
for column in columns_to_encode:
    df_copy[column] = encoder.fit_transform(df_copy[column])

In [7]:
print(df_copy[columns_to_encode].head())

   AccidentArea  Sex  Fault  PoliceReportFiled  WitnessPresent  AgentType
0             1    0      0                  0               0          0
1             1    1      0                  1               0          0
2             1    1      0                  0               0          0
3             0    1      1                  1               0          0
4             1    0      1                  0               0          0


In [8]:
for column in columns_to_encode:
    print(f"{column}: {df_copy[column].unique()}")

AccidentArea: [1 0]
Sex: [0 1]
Fault: [0 1]
PoliceReportFiled: [0 1]
WitnessPresent: [0 1]
AgentType: [0 1]


In [9]:
df_copy.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,1,Tuesday,Jan,1,0,Single,21,0,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,3 years,26 to 30,0,0,0,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,1,Monday,Jan,4,1,Single,34,0,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,6 years,31 to 35,1,0,0,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,1,Thursday,Nov,2,1,Married,47,0,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,7 years,41 to 50,0,0,0,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,0,Friday,Jul,1,1,Married,65,1,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,more than 7,51 to 65,1,0,0,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,1,Tuesday,Feb,2,0,Single,27,1,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,5 years,31 to 35,0,0,0,none,no change,1 vehicle,1994,Collision


## Maping `AgeOfVehicle` to Groups

In [10]:
aov_label = {'2 years': 0, 'more than 7': 0, '7 years': 0, '5 years': 1, '6 years': 1, '3 years': 2, '4 years': 2, 'new': 2 }
df_copy['AgeOfVehicle'] = df_copy['AgeOfVehicle'].map(aov_label)

In [11]:
df_copy.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,1,Tuesday,Jan,1,0,Single,21,0,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,2,26 to 30,0,0,0,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,1,Monday,Jan,4,1,Single,34,0,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,1,31 to 35,1,0,0,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,1,Thursday,Nov,2,1,Married,47,0,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,0,41 to 50,0,0,0,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,0,Friday,Jul,1,1,Married,65,1,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,0,51 to 65,1,0,0,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,1,Tuesday,Feb,2,0,Single,27,1,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,1,31 to 35,0,0,0,none,no change,1 vehicle,1994,Collision


## Mapping `BasePolicy` to Numerical Values

In [12]:
bp_label = {'Liability': 0, 'Collision': 1, 'All Perils': 2}
df_copy['BasePolicy'] = df_copy['BasePolicy'].map(bp_label)

In [13]:
df_copy.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,1,Tuesday,Jan,1,0,Single,21,0,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,2,26 to 30,0,0,0,none,1 year,3 to 4,1994,0
1,Jan,3,Wednesday,Honda,1,Monday,Jan,4,1,Single,34,0,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,1,31 to 35,1,0,0,none,no change,1 vehicle,1994,1
2,Oct,5,Friday,Honda,1,Thursday,Nov,2,1,Married,47,0,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,0,41 to 50,0,0,0,none,no change,1 vehicle,1994,1
3,Jun,2,Saturday,Toyota,0,Friday,Jul,1,1,Married,65,1,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,0,51 to 65,1,0,0,more than 5,no change,1 vehicle,1994,0
4,Jan,5,Monday,Honda,1,Tuesday,Feb,2,0,Single,27,1,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,1,31 to 35,0,0,0,none,no change,1 vehicle,1994,1


## Mapping `VehiclePrice` to Groups

In [14]:
vp_label = {'60000 to 69000': 0, '20000 to 29000': 0,  '30000 to 39000': 0, 'less than 20000': 1, '40000 to 59000': 1, 'more than 69000': 1}
df_copy['VehiclePrice'] = df_copy['VehiclePrice'].map(vp_label)

In [15]:
df_copy.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,1,Tuesday,Jan,1,0,Single,21,0,Sport - Liability,Sport,1,0,1,12,300,1,more than 30,more than 30,none,2,26 to 30,0,0,0,none,1 year,3 to 4,1994,0
1,Jan,3,Wednesday,Honda,1,Monday,Jan,4,1,Single,34,0,Sport - Collision,Sport,1,0,2,15,400,4,more than 30,more than 30,none,1,31 to 35,1,0,0,none,no change,1 vehicle,1994,1
2,Oct,5,Friday,Honda,1,Thursday,Nov,2,1,Married,47,0,Sport - Collision,Sport,1,0,3,7,400,3,more than 30,more than 30,1,0,41 to 50,0,0,0,none,no change,1 vehicle,1994,1
3,Jun,2,Saturday,Toyota,0,Friday,Jul,1,1,Married,65,1,Sedan - Liability,Sport,0,0,4,4,400,2,more than 30,more than 30,1,0,51 to 65,1,0,0,more than 5,no change,1 vehicle,1994,0
4,Jan,5,Monday,Honda,1,Tuesday,Feb,2,0,Single,27,1,Sport - Collision,Sport,1,0,5,3,400,1,more than 30,more than 30,none,1,31 to 35,0,0,0,none,no change,1 vehicle,1994,1


## Dropping Unnecessary Columns

In [16]:
columns_to_drop = ['Month', 'WeekOfMonth', 'DayOfWeek', 'DayOfWeekClaimed', 'WeekOfMonthClaimed', 'PolicyNumber']
df_copy = df_copy.drop(columns=columns_to_drop)

In [17]:
df_copy.shape

(15420, 27)

In [18]:
df_copy.head()

,Make,AccidentArea,MonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Honda,1,Jan,0,Single,21,0,Sport - Liability,Sport,1,0,12,300,1,more than 30,more than 30,none,2,26 to 30,0,0,0,none,1 year,3 to 4,1994,0
1,Honda,1,Jan,1,Single,34,0,Sport - Collision,Sport,1,0,15,400,4,more than 30,more than 30,none,1,31 to 35,1,0,0,none,no change,1 vehicle,1994,1
2,Honda,1,Nov,1,Married,47,0,Sport - Collision,Sport,1,0,7,400,3,more than 30,more than 30,1,0,41 to 50,0,0,0,none,no change,1 vehicle,1994,1
3,Toyota,0,Jul,1,Married,65,1,Sedan - Liability,Sport,0,0,4,400,2,more than 30,more than 30,1,0,51 to 65,1,0,0,more than 5,no change,1 vehicle,1994,0
4,Honda,1,Feb,0,Single,27,1,Sport - Collision,Sport,1,0,3,400,1,more than 30,more than 30,none,1,31 to 35,0,0,0,none,no change,1 vehicle,1994,1


## Converting Columns to Strings

In [19]:
dtype_string = ['RepNumber', 'Deductible', 'Year']
for col in dtype_string:
    df_copy[col] = df_copy[col].astype(str)

## One-Hot Encoding for Categorical Features

In [20]:
encoding_columns = ['Make', 'MonthClaimed', 'MaritalStatus', 'PolicyType', 'VehicleCategory', 'RepNumber', 'Deductible', 'Days_Policy_Accident', 'Days_Policy_Claim', 
                           'PastNumberOfClaims', 'AgeOfPolicyHolder', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year']
df_copy = pd.get_dummies(df_copy, columns=encoding_columns)
df_copy.shape

(15420, 119)

In [21]:
df_copy.head()

,AccidentArea,Sex,Age,Fault,VehiclePrice,FraudFound_P,DriverRating,AgeOfVehicle,PoliceReportFiled,WitnessPresent,AgentType,BasePolicy,Make_Accura,Make_BMW,Make_Chevrolet,Make_Dodge,Make_Ferrari,Make_Ford,Make_Honda,Make_Jaguar,Make_Lexus,Make_Mazda,Make_Mecedes,Make_Mercury,Make_Nisson,Make_Pontiac,Make_Porche,Make_Saab,Make_Saturn,Make_Toyota,Make_VW,MonthClaimed_0,MonthClaimed_Apr,MonthClaimed_Aug,MonthClaimed_Dec,MonthClaimed_Feb,MonthClaimed_Jan,MonthClaimed_Jul,MonthClaimed_Jun,MonthClaimed_Mar,MonthClaimed_May,MonthClaimed_Nov,MonthClaimed_Oct,MonthClaimed_Sep,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single,MaritalStatus_Widow,PolicyType_Sedan - All Perils,PolicyType_Sedan - Collision,PolicyType_Sedan - Liability,PolicyType_Sport - All Perils,PolicyType_Sport - Collision,PolicyType_Sport - Liability,PolicyType_Utility - All Perils,PolicyType_Utility - Collision,PolicyType_Utility - Liability,VehicleCategory_Sedan,VehicleCategory_Sport,VehicleCategory_Utility,RepNumber_1,RepNumber_10,RepNumber_11,RepNumber_12,RepNumber_13,RepNumber_14,RepNumber_15,RepNumber_16,RepNumber_2,RepNumber_3,RepNumber_4,RepNumber_5,RepNumber_6,RepNumber_7,RepNumber_8,RepNumber_9,Deductible_300,Deductible_400,Deductible_500,Deductible_700,Days_Policy_Accident_1 to 7,Days_Policy_Accident_15 to 30,Days_Policy_Accident_8 to 15,Days_Policy_Accident_more than 30,Days_Policy_Accident_none,Days_Policy_Claim_15 to 30,Days_Policy_Claim_8 to 15,Days_Policy_Claim_more than 30,Days_Policy_Claim_none,PastNumberOfClaims_1,PastNumberOfClaims_2 to 4,PastNumberOfClaims_more than 4,PastNumberOfClaims_none,AgeOfPolicyHolder_16 to 17,AgeOfPolicyHolder_18 to 20,AgeOfPolicyHolder_21 to 25,AgeOfPolicyHolder_26 to 30,AgeOfPolicyHolder_31 to 35,AgeOfPolicyHolder_36 to 40,AgeOfPolicyHolder_41 to 50,AgeOfPolicyHolder_51 to 65,AgeOfPolicyHolder_over 65,NumberOfSuppliments_1 to 2,NumberOfSuppliments_3 to 5,NumberOfSuppliments_more than 5,NumberOfSuppliments_none,AddressChange_Claim_1 year,AddressChange_Claim_2 to 3 years,AddressChange_Claim_4 to 8 years,AddressChange_Claim_no change,AddressChange_Claim_under 6 months,NumberOfCars_1 vehicle,NumberOfCars_2 vehicles,NumberOfCars_3 to 4,NumberOfCars_5 to 8,NumberOfCars_more than 8,Year_1994,Year_1995,Year_1996
0,1,0,21,0,1,0,1,2,0,0,0,0,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,True,False,False,True,False,False
1,1,1,34,0,1,0,4,1,1,0,0,1,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False
2,1,1,47,0,1,0,3,0,0,0,0,1,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False

## Identifying and dropping constants 

In [22]:
encoded_columns = [col for col in df_copy.columns if '_' in col]
encoded_columns.remove("FraudFound_P")
constant_features = []
for col in encoded_columns:
    if df_copy[col].sum() <= 5:
        constant_features.append(col)
print(len(constant_features))

9


In [23]:
df_copy.drop(columns=constant_features, axis=1, inplace=True)
df_copy.shape

(15420, 110)

In [24]:
zero_age = df[df['Age'] == 0].shape[0]

print(f"{zero_age}")

320


In [25]:
df_copy['Age'] = df_copy['Age'].apply(lambda x: np.nan if x == 0 else x)
median_age = df_copy['Age'].median()  
df_copy['Age'].fillna(median_age, inplace=True) 

print(df_copy['Age'].isnull().sum())

0


/var/folders/5l/hw_0wn511454x7gg5m0w2wrm0000gn/T/ipykernel_51247/1434974584.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_copy['Age'].fillna(median_age, inplace=True)


In [26]:
df_copy.head()

,AccidentArea,Sex,Age,Fault,VehiclePrice,FraudFound_P,DriverRating,AgeOfVehicle,PoliceReportFiled,WitnessPresent,AgentType,BasePolicy,Make_Accura,Make_BMW,Make_Chevrolet,Make_Dodge,Make_Ford,Make_Honda,Make_Jaguar,Make_Mazda,Make_Mercury,Make_Nisson,Make_Pontiac,Make_Saab,Make_Saturn,Make_Toyota,Make_VW,MonthClaimed_Apr,MonthClaimed_Aug,MonthClaimed_Dec,MonthClaimed_Feb,MonthClaimed_Jan,MonthClaimed_Jul,MonthClaimed_Jun,MonthClaimed_Mar,MonthClaimed_May,MonthClaimed_Nov,MonthClaimed_Oct,MonthClaimed_Sep,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single,MaritalStatus_Widow,PolicyType_Sedan - All Perils,PolicyType_Sedan - Collision,PolicyType_Sedan - Liability,PolicyType_Sport - All Perils,PolicyType_Sport - Collision,PolicyType_Utility - All Perils,PolicyType_Utility - Collision,PolicyType_Utility - Liability,VehicleCategory_Sedan,VehicleCategory_Sport,VehicleCategory_Utility,RepNumber_1,RepNumber_10,RepNumber_11,RepNumber_12,RepNumber_13,RepNumber_14,RepNumber_15,RepNumber_16,RepNumber_2,RepNumber_3,RepNumber_4,RepNumber_5,RepNumber_6,RepNumber_7,RepNumber_8,RepNumber_9,Deductible_300,Deductible_400,Deductible_500,Deductible_700,Days_Policy_Accident_1 to 7,Days_Policy_Accident_15 to 30,Days_Policy_Accident_8 to 15,Days_Policy_Accident_more than 30,Days_Policy_Accident_none,Days_Policy_Claim_15 to 30,Days_Policy_Claim_8 to 15,Days_Policy_Claim_more than 30,PastNumberOfClaims_1,PastNumberOfClaims_2 to 4,PastNumberOfClaims_more than 4,PastNumberOfClaims_none,AgeOfPolicyHolder_16 to 17,AgeOfPolicyHolder_18 to 20,AgeOfPolicyHolder_21 to 25,AgeOfPolicyHolder_26 to 30,AgeOfPolicyHolder_31 to 35,AgeOfPolicyHolder_36 to 40,AgeOfPolicyHolder_41 to 50,AgeOfPolicyHolder_51 to 65,AgeOfPolicyHolder_over 65,NumberOfSuppliments_1 to 2,NumberOfSuppliments_3 to 5,NumberOfSuppliments_more than 5,NumberOfSuppliments_none,AddressChange_Claim_1 year,AddressChange_Claim_2 to 3 years,AddressChange_Claim_4 to 8 years,AddressChange_Claim_no change,NumberOfCars_1 vehicle,NumberOfCars_2 vehicles,NumberOfCars_3 to 4,NumberOfCars_5 to 8,Year_1994,Year_1995,Year_1996
0,1,0,21.0,0,1,0,1,2,0,0,0,0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,True,False,True,False,False
1,1,1,34.0,0,1,0,4,1,1,0,0,1,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True,True,False,False,False,True,False,False
2,1,1,47.0,0,1,0,3,0,0,0,0,1,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,True,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,True,False,False,False,True,False,False
3,0,1,65.0,1,0,0,2,0,1,0,0,0,False,False,False,False,False,False,False,False,False,False,False,False,Fal

## Exporting the processed data to .CSV

In [27]:
file_name = 'processed_data.csv'
file_path = os.path.join(folder_name, file_name)

df_copy.to_csv(file_path, index=False)

print(f"File saved to {file_path}")

File saved to Data/processed_data.csv


In [28]:
df_copy1 = df_copy.copy()

In [29]:
df_copy1.shape

(15420, 110)

In [30]:
X = df_copy1.drop(columns="FraudFound_P")
y = df_copy1["FraudFound_P"]

In [31]:
X.shape

(15420, 109)

## Handling Imbalanced Data with SMOTE

In [32]:
smote = SMOTE(random_state=0)
X_smote, y_smote = smote.fit_resample(X, y)

In [33]:
print(pd.Series(y_smote).value_counts())

FraudFound_P
0    14497
1    14497
Name: count, dtype: int64


In [34]:
df_smote = pd.concat([X_smote, y_smote], axis=1)
df_smote.shape

(28994, 110)

In [35]:
X_smote = df_smote.drop(columns="FraudFound_P")
y_smote = df_smote["FraudFound_P"]

## Variance Threshold (Low-variance features are removed based on a threshold of 0.01.)

In [36]:
selector = VarianceThreshold(threshold=0.01)
X_reduced = selector.fit_transform(X_smote)
X_reduced.shape

(28994, 85)

In [37]:
selected_feature_indices = selector.get_support(indices=True)
selected_feature_names = X_smote.columns[selected_feature_indices]
X_reduced = pd.DataFrame(X_reduced, columns=selected_feature_names)

In [38]:
X_reduced.head()

,AccidentArea,Sex,Age,Fault,VehiclePrice,DriverRating,AgeOfVehicle,PoliceReportFiled,BasePolicy,Make_Accura,Make_Chevrolet,Make_Ford,Make_Honda,Make_Mazda,Make_Pontiac,Make_Saab,Make_Toyota,Make_VW,MonthClaimed_Apr,MonthClaimed_Aug,MonthClaimed_Dec,MonthClaimed_Feb,MonthClaimed_Jan,MonthClaimed_Jul,MonthClaimed_Jun,MonthClaimed_Mar,MonthClaimed_May,MonthClaimed_Nov,MonthClaimed_Oct,MonthClaimed_Sep,MaritalStatus_Married,MaritalStatus_Single,PolicyType_Sedan - All Perils,PolicyType_Sedan - Collision,PolicyType_Sedan - Liability,PolicyType_Sport - Collision,PolicyType_Utility - All Perils,VehicleCategory_Sedan,VehicleCategory_Sport,VehicleCategory_Utility,RepNumber_1,RepNumber_10,RepNumber_11,RepNumber_12,RepNumber_13,RepNumber_14,RepNumber_15,RepNumber_16,RepNumber_2,RepNumber_3,RepNumber_4,RepNumber_5,RepNumber_6,RepNumber_7,RepNumber_8,RepNumber_9,Deductible_400,Deductible_500,Deductible_700,PastNumberOfClaims_1,PastNumberOfClaims_2 to 4,PastNumberOfClaims_more than 4,PastNumberOfClaims_none,AgeOfPolicyHolder_16 to 17,AgeOfPolicyHolder_21 to 25,AgeOfPolicyHolder_26 to 30,AgeOfPolicyHolder_31 to 35,AgeOfPolicyHolder_36 to 40,AgeOfPolicyHolder_41 to 50,AgeOfPolicyHolder_51 to 65,AgeOfPolicyHolder_over 65,NumberOfSuppliments_1 to 2,NumberOfSuppliments_3 to 5,NumberOfSuppliments_more than 5,NumberOfSuppliments_none,AddressChange_Claim_1 year,AddressChange_Claim_2 to 3 years,AddressChange_Claim_4 to 8 years,AddressChange_Claim_no change,NumberOfCars_1 vehicle,NumberOfCars_2 vehicles,NumberOfCars_3 to 4,Year_1994,Year_1995,Year_1996
0,1.0,0.0,21.0,0.0,1.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
1,1.0,1.0,34.0,0.0,1.0,4.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,1.0,47.0,0.0,1.0,3.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.0,1.0,65.0,1.0,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
4,1.0,0.0,27.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0


In [39]:
X_reduced.shape

(28994, 85)

## Recursive Feature Elimination with Cross-Validation (RFECV)

In [40]:
lightgbm_model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
rfecv = RFECV(estimator=lightgbm_model, step=1, cv=StratifiedKFold(5), scoring='roc_auc', n_jobs=-1)
rfecv.fit(X_reduced, y_smote)

RFECV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
      estimator=LGBMClassifier(random_state=42, verbose=-1), n_jobs=-1,
      scoring='roc_auc')

In [41]:
optimal_num_features =rfecv.n_features_
X_selected = rfecv.transform(X_reduced)
selected_feature_indices = np.where(rfecv.support_)[0]
selected_feature_names = X_reduced.columns[selected_feature_indices]
X_rfecv = pd.DataFrame(X_selected, columns=selected_feature_names)
X_rfecv['FraudFound_P'] = y_smote

In [42]:
X_rfecv.shape

(28994, 71)

In [43]:
X_rfecv.head()

,AccidentArea,Age,Fault,VehiclePrice,DriverRating,AgeOfVehicle,BasePolicy,Make_Accura,Make_Chevrolet,Make_Ford,Make_Honda,Make_Mazda,Make_Pontiac,Make_Saab,Make_Toyota,MonthClaimed_Apr,MonthClaimed_Aug,MonthClaimed_Dec,MonthClaimed_Feb,MonthClaimed_Jan,MonthClaimed_Jul,MonthClaimed_Jun,MonthClaimed_Mar,MonthClaimed_May,MonthClaimed_Nov,MonthClaimed_Oct,MonthClaimed_Sep,MaritalStatus_Married,MaritalStatus_Single,PolicyType_Sedan - All Perils,PolicyType_Sport - Collision,PolicyType_Utility - All Perils,VehicleCategory_Sedan,VehicleCategory_Utility,RepNumber_1,RepNumber_10,RepNumber_11,RepNumber_12,RepNumber_13,RepNumber_14,RepNumber_15,RepNumber_16,RepNumber_2,RepNumber_3,RepNumber_4,RepNumber_5,RepNumber_6,RepNumber_7,RepNumber_8,RepNumber_9,Deductible_400,Deductible_500,Deductible_700,PastNumberOfClaims_1,PastNumberOfClaims_2 to 4,PastNumberOfClaims_more than 4,PastNumberOfClaims_none,AgeOfPolicyHolder_16 to 17,AgeOfPolicyHolder_31 to 35,NumberOfSuppliments_1 to 2,NumberOfSuppliments_3 to 5,NumberOfSuppliments_more than 5,NumberOfSuppliments_none,AddressChange_Claim_2 to 3 years,NumberOfCars_1 vehicle,NumberOfCars_2 vehicles,NumberOfCars_3 to 4,Year_1994,Year_1995,Year_1996,FraudFound_P
0,1.0,21.0,0.0,1.0,1.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0
1,1.0,34.0,0.0,1.0,4.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0
2,1.0,47.0,0.0,1.0,3.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0
3,0.0,65.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0
4,1.0,27.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0


In [44]:
X_rfecv['FraudFound_P'].value_counts()

FraudFound_P
0    14497
1    14497
Name: count, dtype: int64

## Exporting the processed data to .CSV

In [45]:
file_name = 'processed_data_smote.csv'
file_path = os.path.join(folder_name, file_name)

X_rfecv.to_csv(file_path, index=False)

print(f"File saved to {file_path}")

File saved to Data/processed_data_smote.csv
